# Last.fm Album Recommendations

Similar-album recommendations via Last.fm's `track.getSimilar` API.

Compare three ways of picking the seed track from the album:

- **(a) Top listener** — the track with the most Last.fm listeners on the album
- **(b) Random** — a random track from the album tracklist
- **(c) All-tracks overlap** — run similarity for every album track and return albums recommended by multiple tracks; falls back to **(a)** if there is no overlap

All paths exclude albums by the seed artist. API responses are cached per run.

Set your API key in `bench/.env`:

```
LASTFM_API_KEY=your_key_here
```

In [12]:
import importlib

import pandas as pd

import lastfm_albums

importlib.reload(lastfm_albums)

from lastfm_albums import compare_recommendations, get_album_info

## Seed album

In [13]:
SEED_ALBUM = "OK Computer"
SEED_ARTIST = "Radiohead"
N = 5
RANDOM_SEED = 42

seed = get_album_info(SEED_ARTIST, SEED_ALBUM)
pd.Series({k: seed[k] for k in seed if k != "tracks"})

key                                   radiohead::ok computer
artist                                             Radiohead
album                                            OK Computer
mbid                    0b6b4ba0-d36f-47bd-b4ea-6a5b91842d29
playcount                                          260472466
listeners                                            4828347
url          https://www.last.fm/music/Radiohead/OK+Computer
dtype: object

## Compare seed-track strategies

In [14]:
def show_strategy(title, seed_track, recommendations, extra_cols=None):
    print(title)
    if seed_track is not None:
        print(f"Seed track: {seed_track['name']}", end="")
        if "listeners" in seed_track:
            print(f" ({seed_track['listeners']:,} listeners)")
        else:
            print()
    if recommendations.empty:
        print("(no results)")
        return
    cols = ["album", "artist", "similarity_score", "matched_via", "url"]
    if extra_cols:
        cols = extra_cols + cols
    display(recommendations[cols])


results = compare_recommendations(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n=N,
    random_seed=RANDOM_SEED,
)
seed = results["seed"]

print(f"Seed album: {seed['album']} — {seed['artist']}")
print(f"Tracks on album: {len(seed['tracks'])}")
print(f"API calls this run: {results['api_calls']}")
print()

show_strategy(
    "(a) Top-listener track",
    results["top_listener_track"],
    results["top_listener_recs"],
)
print()
show_strategy(
    "(b) Random track",
    results["random_track"],
    results["random_track_recs"],
)
print()

all_tracks_title = "(c) All-tracks overlap"
if results["all_tracks_used_fallback"]:
    all_tracks_title += " (no overlap — fell back to top-listener track)"
show_strategy(
    all_tracks_title,
    results["all_tracks_fallback_track"],
    results["all_tracks_recs"],
    extra_cols=["vote_count"] if "vote_count" in results["all_tracks_recs"].columns else None,
)
print()

print("Albums recommended by both (a) and (b):")
display(results["pairwise_overlap"])

Seed album: OK Computer — Radiohead
Tracks on album: 12
API calls this run: 48

(a) Top-listener track
Seed track: No Surprises (3,412,517 listeners)


,album,artist,similarity_score,matched_via,url
0,Surfer Rosa,Pixies,0.275238,track 'Where Is My Mind?' similar to 'No Surpr...,https://www.last.fm/music/Pixies/Surfer+Rosa
1,The Colour and the Shape,Foo Fighters,0.249315,track 'Everlong' similar to 'No Surprises',https://www.last.fm/music/Foo+Fighters/The+Col...
2,Plastic Beach,Gorillaz,0.225677,track 'On Melancholy Hill' similar to 'No Surp...,https://www.last.fm/music/Gorillaz/Plastic+Beach
3,Weezer,Weezer,0.208867,track 'Buddy Holly' similar to 'No Surprises',https://www.last.fm/music/Weezer/Weezer
4,B-Sides & Rarities,Deftones,0.203997,track 'Change (In the House of Flies)' similar...,https://www.last.fm/music/Deftones/B-Sides+&+R...



(b) Random track
Seed track: Lucky


,album,artist,similarity_score,matched_via,url
0,Grace,Jeff Buckley,0.162629,track 'Grace' similar to 'Lucky',https://www.last.fm/music/Jeff+Buckley/Grace
1,Surfer Rosa,Pixies,0.138759,track 'Where Is My Mind?' similar to 'Lucky',https://www.last.fm/music/Pixies/Surfer+Rosa
2,Origin of Symmetry,Muse,0.125322,track 'Space Dementia' similar to 'Lucky',https://www.last.fm/music/Muse/Origin+of+Symmetry
3,Doolittle,Pixies,0.117033,track 'Here Comes Your Man' similar to 'Lucky',https://www.last.fm/music/Pixies/Doolittle
4,Twilight,bôa,0.116993,track 'Twilight' similar to 'Lucky',https://www.last.fm/music/b%C3%B4a/Twilight



(c) All-tracks overlap


,vote_count,album,artist,similarity_score,matched_via,url
0,12,Surfer Rosa,Pixies,0.326701,"overlap from 12 tracks: Airbag, Paranoid Andro...",https://www.last.fm/music/Pixies/Surfer+Rosa
1,10,Grace,Jeff Buckley,0.218816,"overlap from 10 tracks: Airbag, Paranoid Andro...",https://www.last.fm/music/Jeff+Buckley/Grace
2,8,Absolution,Muse,0.236321,"overlap from 8 tracks: Airbag, Paranoid Androi...",https://www.last.fm/music/Muse/Absolution
3,8,Weezer,Weezer,0.234870,"overlap from 8 tracks: Airbag, Subterranean Ho...",https://www.last.fm/music/Weezer/Weezer
4,7,The Colour and the Shape,Foo Fighters,0.304185,"overlap from 7 tracks: Paranoid Android, Exit ...",https://www.last.fm/music/Foo+Fighters/The+Col...



Albums recommended by both (a) and (b):


,album_top_listener,artist_top_listener,seed_track_top_listener,similarity_score_top_listener,similarity_score_random
0,Surfer Rosa,Pixies,No Surprises,0.275238,0.138759


In [ ]:
SEED_ALBUM = "Acabou Chorare"
SEED_ARTIST = "Novos Baianos"

results = compare_recommendations(SEED_ALBUM, artist=SEED_ARTIST, n=N, random_seed=RANDOM_SEED)
show_strategy("(a) Top-listener track", results["top_listener_track"], results["top_listener_recs"])
print()
show_strategy("(b) Random track", results["random_track"], results["random_track_recs"])
print()
title = "(c) All-tracks overlap"
if results["all_tracks_used_fallback"]:
    title += " (fell back to top-listener)"
show_strategy(
    title,
    results["all_tracks_fallback_track"],
    results["all_tracks_recs"],
    extra_cols=["vote_count"] if "vote_count" in results["all_tracks_recs"].columns else None,
)
print()
display(results["pairwise_overlap"])